In [ ]:
# ============================================================
# SETUP — Install required packages
# Run this cell first if packages are missing
# imbalanced-learn 0.12+ required for sklearn 1.4+
# ============================================================

import subprocess, sys

packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "imbalanced-learn>=0.12.0",  # must be 0.12+ for sklearn 1.4+
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed successfully")

# FIXED Credit Risk Model — Full Pipeline
## Hackathon Submission

**All 14 bugs identified and fixed. Full EDA → Cleaning → Preprocessing → Modeling → Evaluation**

| Bug # | Description | Fix |
|-------|-------------|-----|
| 1 | SimpleImputer wrong import | Moved to sklearn.impute |
| 2 | Wrong file path | Fixed to credit_risk_dataset.csv |
| 3 | Data leakage post-outcome cols | Dropped before any processing |
| 4 | Feature selection on full data | Done after split on train only |
| 5 | Preprocessor fit on train+test | fit on train, transform test |
| 6 | Hyperparameter tuning on test set | GridSearchCV with StratifiedKFold |
| 7 | Threshold tuning on test set | Tuned on validation fold |
| 8 | Normalization stats from full data | Computed on train only |
| 9 | More leakage features section 3 | All post-outcome features dropped |
| 10 | Wrong column name annual_income | Corrected to annual_inc |
| 11 | auc variable shadows auc function | Renamed to auc_score |
| 12 | Missing metric imports | Added precision_score recall_score brier_score_loss |
| 13 | Feature importance mismatch after OHE | Used get_feature_names_out |
| 14 | Undefined tp_y fn_y variables | Replaced with recall_score on subset |

In [ ]:
# ============================================================
# CELL 1 — IMPORTS
# BUG #1  FIXED: SimpleImputer was imported from sklearn.preprocessing
#                Correct module is sklearn.impute
# BUG #12 FIXED: precision_score, recall_score, brier_score_loss
#                were used later but never imported — added here
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer          # FIX #1: was sklearn.preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score,    # FIX #12: were missing
    confusion_matrix, ConfusionMatrixDisplay,
    brier_score_loss, classification_report      # FIX #12: were missing
)
from imblearn.over_sampling import SMOTE
from collections import Counter

np.random.seed(42)
print('All imports successful')

---
## STEP 1 — Load Data

In [ ]:
# ============================================================
# CELL 2 — LOAD DATA
# BUG #2 FIXED: Original path was '../../datasets/_credit_risk_dataset.csv'
#               Wrong directory AND wrong filename (extra underscore prefix)
# ============================================================

df = pd.read_csv('credit_risk_dataset.csv')    # FIX #2: corrected path

print(f'Shape   : {df.shape}')
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print(f'\nAll columns:')
print(df.columns.tolist())
df.head()

---
## STEP 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# CELL 3 — BASIC INFO AND DATA TYPES
# ============================================================

print('='*60)
print('DATA TYPES AND NON-NULL COUNTS')
print('='*60)
df.info()

print('\n' + '='*60)
print('STATISTICAL SUMMARY')
print('='*60)
df.describe().round(2)

In [ ]:
# ============================================================
# CELL 4 — TARGET VARIABLE ANALYSIS
# Dataset is heavily imbalanced: 96% no-default, 4% default
# This means accuracy is a completely misleading metric!
# ROC-AUC and Recall are the right metrics for this problem
# ============================================================

target_counts = df['target_flag'].value_counts()
target_pct    = df['target_flag'].value_counts(normalize=True) * 100

print('='*50)
print('TARGET VARIABLE DISTRIBUTION')
print('='*50)
print(f'  No Default (0): {target_counts[0]:,}  ({target_pct[0]:.1f}%)')
print(f'  Default    (1): {target_counts[1]:,}  ({target_pct[1]:.1f}%)')
print(f'\nIMBALANCED DATASET: only {target_pct[1]:.1f}% defaults!')
print('A model predicting all zeros gets 96% accuracy automatically.')
print('So accuracy is MISLEADING here. Use ROC-AUC instead.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['No Default (0)', 'Default (1)'], target_counts.values,
            color=['steelblue', 'tomato'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Class Count', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=11)

axes[1].pie(target_counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Proportion', fontsize=12, fontweight='bold')

plt.suptitle('Class Imbalance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 5 — MISSING VALUE ANALYSIS
# ============================================================

missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %'    : missing_pct.round(2)
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print('='*50)
print('MISSING VALUE REPORT')
print('='*50)
print(missing_df)
print(f'\nStrategy: Impute numeric with MEDIAN, categorical with MODE')
print('Both done inside the pipeline AFTER train-test split')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing_df.index, missing_df['Missing %'],
               color='coral', edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Feature', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6 — OUTLIER DETECTION
# Found: person_age max = 999 (impossible!)
#        annual_inc max = 1.2M (extreme outlier)
# ============================================================

print('='*55)
print('OUTLIER DETECTION')
print('='*55)
num_cols_check = ['person_age', 'annual_inc', 'loan_amt', 'interest_rate', 'credit_score']

for col in num_cols_check:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lo  = q1 - 1.5 * iqr
    hi  = q3 + 1.5 * iqr
    out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col:20s}  min={df[col].min():.0f}  max={df[col].max():.0f}  outliers={out}')

print(f'\nALERT: person_age max = {df["person_age"].max()} — impossible, will cap at 100')
print(f'ALERT: annual_inc  max = {df["annual_inc"].max():,.0f} — extreme, will cap at 99th pct')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['person_age', 'annual_inc', 'credit_score']):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='navy'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xticks([])
plt.suptitle('Outlier Boxplots', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 7 — CATEGORICAL COLUMN ANALYSIS
# Found: employment_type and residence_type have case issues
# 'employed' and 'Employed' are the SAME category!
# ============================================================

cat_cols = ['home_ownership', 'loan_intent', 'loan_grade', 'employment_type', 'residence_type']

print('='*60)
print('CATEGORICAL UNIQUE VALUES')
print('='*60)
for col in cat_cols:
    print(f'{col:20s}: {df[col].unique().tolist()}')

print('\nBUG: employment_type has "employed" and "Employed" — same thing!')
print('BUG: residence_type has "Rural" and "RURAL" — same thing!')
print('FIX: Standardize all to uppercase in cleaning step')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    temp = df[col].str.upper()
    rate = df.groupby(temp)['target_flag'].mean().sort_values(ascending=False)
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(rate)))
    bars = axes[i].bar(rate.index, rate.values * 100, color=colors,
                       edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'Default Rate by {col}', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Default %')
    axes[i].tick_params(axis='x', rotation=30)
    for bar in bars:
        h = bar.get_height()
        axes[i].text(bar.get_x() + bar.get_width()/2, h + 0.2,
                     f'{h:.1f}%', ha='center', fontsize=8)

axes[5].axis('off')
plt.suptitle('Default Rate by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 8 — NUMERIC FEATURE DISTRIBUTIONS vs TARGET
# ============================================================

num_cols = ['person_age', 'annual_inc', 'loan_amt', 'interest_rate',
            'credit_score', 'employment_length', 'income_ratio', 'monthly_income']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color, name in [(0, 'steelblue', 'No Default'), (1, 'tomato', 'Default')]:
        data = df[df['target_flag'] == label][col].dropna()
        axes[i].hist(data, bins=30, alpha=0.6, color=color, label=name, density=True)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Numeric Feature Distributions by Default Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9 — CORRELATION HEATMAP (legitimate features only)
# We EXCLUDE leakage columns from this analysis
# to avoid being misled by artificially high correlations
# ============================================================

clean_num = ['person_age', 'annual_inc', 'employment_length', 'loan_amt',
             'interest_rate', 'credit_score', 'monthly_income', 'income_ratio',
             'target_flag']

corr_matrix = df[clean_num].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9})
ax.set_title('Correlation Matrix (Legitimate Features Only)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

target_corr = corr_matrix['target_flag'].drop('target_flag').sort_values(key=abs, ascending=False)
print('Top correlations with target_flag (no leakage):')
print(target_corr.round(3))

In [ ]:
# ============================================================
# CELL 10 — COLUMN AUDIT: IDENTIFY WHAT TO DROP
# BUG #3 + #9 DOCUMENTED HERE
# ============================================================

print('='*60)
print('COLUMN AUDIT')
print('='*60)

print('\nDROP — LEAKAGE (post-outcome, not available at decision time):')
print('  loan_status_final   — recorded after loan closes')
print('  repayment_flag      — recorded after loan closes')
print('  last_payment_status — recorded after loan closes')

print('\nDROP — NOISE / USELESS:')
dup_match = (df['duplicate_feature'] == df['person_age']).mean()
print(f'  duplicate_feature — {dup_match:.1%} identical to person_age')
print(f'  random_score_1    — corr={df["random_score_1"].corr(df["target_flag"]):.4f} (noise)')
print(f'  random_score_2    — corr={df["random_score_2"].corr(df["target_flag"]):.4f} (noise)')

print('\nKEEP — LEGITIMATE FEATURES (available at loan application):')
keep = ['person_age','annual_inc','home_ownership','employment_length',
        'loan_intent','loan_grade','loan_amt','interest_rate',
        'income_ratio','employment_type','residence_type',
        'credit_score','monthly_income']
print(keep)

---
## STEP 3 — Data Cleaning

In [ ]:
# ============================================================
# CELL 11 — DATA CLEANING
# BUG #3  FIXED: Drop all post-outcome leakage columns
# BUG #9  FIXED: Drop additional leakage-derived features
# BUG #10 FIXED: Column was 'annual_income' in broken code
#                Correct name is 'annual_inc'
# ============================================================

df_clean = df.copy()

# Drop leakage and noise columns
drop_cols = [
    'loan_status_final',    # FIX #3: post-outcome leakage
    'repayment_flag',       # FIX #3: post-outcome leakage
    'last_payment_status',  # FIX #3: post-outcome leakage
    'random_score_1',       # noise
    'random_score_2',       # noise
    'duplicate_feature',    # duplicate of person_age
]
df_clean.drop(columns=drop_cols, inplace=True)
print(f'Dropped {len(drop_cols)} columns. Shape now: {df_clean.shape}')

# Fix categorical case inconsistency
df_clean['employment_type'] = df_clean['employment_type'].str.upper().str.replace('-','_')
df_clean['employment_type'] = df_clean['employment_type'].replace({'SELF_EMP':'SELF_EMPLOYED'})
df_clean['residence_type']  = df_clean['residence_type'].str.upper()
print('Fixed case inconsistency in employment_type and residence_type')

# Cap extreme outliers
df_clean['person_age'] = df_clean['person_age'].clip(upper=100)
print(f'person_age capped at 100. New max: {df_clean["person_age"].max()}')

cap_99 = df_clean['annual_inc'].quantile(0.99)  # FIX #10: correct col name annual_inc
df_clean['annual_inc'] = df_clean['annual_inc'].clip(upper=cap_99)
print(f'annual_inc capped at 99th pct ({cap_99:,.0f})')

print(f'\nCleaning complete. Final shape: {df_clean.shape}')

---
## STEP 4 — Feature Engineering (Legitimate Only)

In [ ]:
# ============================================================
# CELL 12 — FEATURE ENGINEERING
# Only features derivable from data available at application time
# No post-outcome columns used
# BUG #8 NOTE: Normalization (income_zscore) was computed on
#              full data in broken code. Here we let the pipeline
#              handle scaling AFTER the split using train stats only
# ============================================================

# Debt-to-income ratio (standard credit risk feature)
df_clean['debt_to_income']   = df_clean['loan_amt'] / (df_clean['annual_inc'] + 1)

# Credit worthiness relative to loan size
df_clean['credit_loan_ratio'] = df_clean['credit_score'] / (df_clean['loan_amt'] + 1)

# Monthly burden
df_clean['loan_to_monthly']   = df_clean['loan_amt'] / (df_clean['monthly_income'] + 1)

print('Engineered features added:')
print('  debt_to_income    = loan_amt / annual_inc')
print('  credit_loan_ratio = credit_score / loan_amt')
print('  loan_to_monthly   = loan_amt / monthly_income')
print(f'\nFinal shape: {df_clean.shape}')
df_clean.head(3)

---
## STEP 5 — Train / Test Split

In [ ]:
# ============================================================
# CELL 13 — TRAIN / TEST SPLIT
# BUG #4 FIXED: In broken code, feature selection and normalization
#               stats were computed BEFORE the split.
#               Correct: split first, everything else after.
# BUG #8 FIXED: global_mean_income was computed on full df.
#               Now handled inside pipeline using train stats only.
# ============================================================

X = df_clean.drop(columns=['target_flag'])
y = df_clean['target_flag']

# Split FIRST — all processing happens after
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves 4% default rate in both sets
)

print('='*50)
print('TRAIN / TEST SPLIT')
print('='*50)
print(f'Train : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test  : {X_test.shape[0]:,}  rows ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain default rate : {y_train.mean():.4f}')
print(f'Test  default rate : {y_test.mean():.4f}')
print('\nBoth sets have same default rate — stratification worked!')

---
## STEP 6 — Preprocessing Pipeline

In [ ]:
# ============================================================
# CELL 14 — BUILD PREPROCESSING PIPELINE
# ============================================================

numeric_features = [
    'person_age', 'annual_inc', 'employment_length', 'loan_amt',
    'interest_rate', 'credit_score', 'monthly_income', 'income_ratio',
    'debt_to_income', 'credit_loan_ratio', 'loan_to_monthly'
]

categorical_features = [
    'home_ownership', 'loan_intent', 'loan_grade',
    'employment_type', 'residence_type'
]

# Numeric: impute missing with median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical: impute missing with mode, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Preprocessing pipeline created')
print(f'  Numeric  : {len(numeric_features)} features')
print(f'  Categorical: {len(categorical_features)} features (will expand after OHE)')

In [ ]:
# ============================================================
# CELL 15 — FIT PREPROCESSOR ON TRAIN ONLY
# BUG #5 FIXED: Broken code did:
#   X_combined = concat([X_train, X_test])
#   preprocessor.fit_transform(X_combined)   <-- WRONG
# This leaks test set statistics into the scaler/imputer.
# FIX: fit on train only, transform test separately.
# ============================================================

X_train_proc = preprocessor.fit_transform(X_train)   # fit + transform on TRAIN
X_test_proc  = preprocessor.transform(X_test)        # transform ONLY on test

print('='*50)
print('PREPROCESSING COMPLETE')
print('='*50)
print(f'X_train_proc shape: {X_train_proc.shape}')
print(f'X_test_proc  shape: {X_test_proc.shape}')
print('\nPreprocessor fitted on TRAIN only')
print('Test set transformed using TRAIN statistics')
print('No contamination between train and test!')

---
## STEP 7 — Handle Class Imbalance with SMOTE

In [ ]:
# ============================================================
# CELL 16 — SMOTE
# SMOTE generates synthetic minority-class samples.
# Applied ONLY to training data.
# Test set stays untouched with real 4% imbalance.
# ============================================================

smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy=0.3)
X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)

print('='*50)
print('SMOTE — CLASS BALANCING')
print('='*50)
print(f'Before SMOTE (train): {Counter(y_train)}')
print(f'After  SMOTE (train): {Counter(y_train_bal)}')
print(f'Test  (unchanged)   : {Counter(y_test)}')
print(f'\nDefault rate after SMOTE: {y_train_bal.mean():.1%}')
print('SMOTE only on TRAINING. Test stays real-world imbalanced.')

---
## STEP 8 — Model Training

In [ ]:
# ============================================================
# CELL 17 — MODEL 1: LOGISTIC REGRESSION (Baseline)
# ============================================================

print('Training Logistic Regression...')

lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_bal, y_train_bal)

lr_cv_proba = cross_val_predict(
    lr_model, X_train_bal, y_train_bal,
    cv=StratifiedKFold(5), method='predict_proba'
)[:, 1]
lr_cv_auc = roc_auc_score(y_train_bal, lr_cv_proba)

print(f'Logistic Regression  CV AUC (train): {lr_cv_auc:.4f}')
print('Model 1 done')

In [ ]:
# ============================================================
# CELL 18 — MODEL 2: RANDOM FOREST WITH CROSS-VALIDATED TUNING
# BUG #6 FIXED: Broken code looped over test set to pick best params
#               (15 models evaluated on y_test = p-hacking)
# FIX: GridSearchCV with StratifiedKFold on TRAINING data only
# ============================================================

print('Training Random Forest with GridSearchCV...')
print('(Takes 1-2 mins)')

param_grid_rf = {
    'max_depth'       : [8, 12, 16],
    'min_samples_leaf': [5, 10]
}

rf_gs = GridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    param_grid_rf,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),  # FIX #6
    scoring='roc_auc',
    n_jobs=-1
)
rf_gs.fit(X_train_bal, y_train_bal)   # TRAIN only — test set never touched

rf_best = rf_gs.best_estimator_
print(f'Best params : {rf_gs.best_params_}')
print(f'Best CV AUC : {rf_gs.best_score_:.4f}')
print('Model 2 done (tuned on train only)')

In [ ]:
# ============================================================
# CELL 19 — MODEL 3: GRADIENT BOOSTING
# ============================================================

print('Training Gradient Boosting...')

param_grid_gb = {'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

gb_gs = GridSearchCV(
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    param_grid_gb,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1
)
gb_gs.fit(X_train_bal, y_train_bal)

gb_best = gb_gs.best_estimator_
print(f'Best params : {gb_gs.best_params_}')
print(f'Best CV AUC : {gb_gs.best_score_:.4f}')
print('Model 3 done')

In [ ]:
# ============================================================
# CELL 20 — MODEL COMPARISON
# BUG #11 FIXED: original code had 'auc = roc_auc_score(...)'
#                which shadows the imported auc() function.
#                Fixed: renamed variable to auc_score
# ============================================================

models = {
    'Logistic Regression': lr_model,
    'Random Forest'      : rf_best,
    'Gradient Boosting'  : gb_best
}

print('='*55)
print('MODEL COMPARISON — Cross-Validated AUC on Train')
print('='*55)
print(f'{"Model":<25}  {"CV AUC"}')
print('-'*45)

cv_aucs = {}
for name, model in models.items():
    proba = cross_val_predict(
        model, X_train_bal, y_train_bal,
        cv=StratifiedKFold(5), method='predict_proba'
    )[:, 1]
    auc_score = roc_auc_score(y_train_bal, proba)   # FIX #11: renamed from auc
    cv_aucs[name] = auc_score
    print(f'{name:<25}  {auc_score:.4f}')

best_model_name = max(cv_aucs, key=cv_aucs.get)
best_model      = models[best_model_name]
print(f'\nBest model: {best_model_name} (AUC = {cv_aucs[best_model_name]:.4f})')

---
## STEP 9 — Threshold Tuning (Validation Only)

In [ ]:
# ============================================================
# CELL 21 — THRESHOLD TUNING
# BUG #7 FIXED: Broken code swept thresholds on y_test directly
#               This is test-set p-hacking.
# FIX: Use cross-val predictions on TRAIN to find best threshold
#      Then apply that threshold ONCE to the test set
# ============================================================

# Get out-of-fold validation probabilities from TRAINING data
val_proba = cross_val_predict(
    best_model, X_train_bal, y_train_bal,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    method='predict_proba'
)[:, 1]

best_threshold = 0.5
best_val_f1    = 0
thr_results    = []

for t in np.arange(0.1, 0.9, 0.02):
    preds = (val_proba >= t).astype(int)
    f1_t  = f1_score(y_train_bal, preds, zero_division=0)
    thr_results.append({'threshold': round(t, 2), 'f1': f1_t})
    if f1_t > best_val_f1:
        best_val_f1   = f1_t
        best_threshold = round(t, 2)

thr_df = pd.DataFrame(thr_results)
print(f'Best threshold (from validation): {best_threshold}')
print(f'Best validation F1             : {best_val_f1:.4f}')
print('\nThreshold tuned on VALIDATION — test set still untouched!')

plt.figure(figsize=(9, 4))
plt.plot(thr_df['threshold'], thr_df['f1'], color='steelblue', linewidth=2)
plt.axvline(best_threshold, color='red', linestyle='--',
            label=f'Best threshold = {best_threshold}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('Threshold vs F1 (Validation Data)', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## STEP 10 — Final Evaluation on Test Set

In [ ]:
# ============================================================
# CELL 22 — FINAL METRICS
# Test set used ONCE here at the very end — correct!
# BUG #11 FIXED: variable named auc_score (not auc)
# BUG #12 FIXED: precision_score, recall_score, brier_score_loss imported
# ============================================================

y_proba = best_model.predict_proba(X_test_proc)[:, 1]
y_pred  = (y_proba >= best_threshold).astype(int)

auc_score = roc_auc_score(y_test, y_proba)                      # FIX #11
accuracy  = (y_pred == y_test).mean()
prec      = precision_score(y_test, y_pred, zero_division=0)    # FIX #12
rec       = recall_score(y_test, y_pred, zero_division=0)       # FIX #12
f1        = f1_score(y_test, y_pred, zero_division=0)
brier     = brier_score_loss(y_test, y_proba)                   # FIX #12
cm        = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('='*60)
print(f'FINAL MODEL PERFORMANCE — {best_model_name}')
print('='*60)
print(f'  ROC-AUC   : {auc_score:.4f}  <-- PRIMARY METRIC')
print(f'  Recall    : {rec:.4f}  <-- % defaults caught')
print(f'  Precision : {prec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Accuracy  : {accuracy:.4f}  (misleading for imbalanced data!)')
print(f'  Brier     : {brier:.4f}')
print(f'  Threshold : {best_threshold}')

print(f'\nCONFUSION MATRIX')
print(f'                Predicted 0  Predicted 1')
print(f'  Actual 0  :   {tn:9d}  {fp:11d}  (TN / FP)')
print(f'  Actual 1  :   {fn:9d}  {tp:11d}  (FN / TP)')

In [ ]:
# ============================================================
# CELL 23 — CONFUSION MATRIX + ROC CURVE PLOTS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No Default (0)', 'Default (1)']
)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'ROC Curve (AUC = {auc_score:.4f})')
axes[1].plot([0,1],[0,1],'k--', lw=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle(f'Model Evaluation — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Default','Default']))

In [ ]:
# ============================================================
# CELL 24 — FEATURE IMPORTANCE
# BUG #13 FIXED: Original used feature_names = numeric + categorical
#                but after OHE, categoricals expand to many columns.
#                Slicing importances to original list gives WRONG names.
# FIX: Use get_feature_names_out() to get the true expanded list
# ============================================================

# Get exact OHE feature names from fitted preprocessor
ohe_names = (
    preprocessor
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(categorical_features)   # FIX #13
)
all_feature_names = numeric_features + list(ohe_names)

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    n = min(len(all_feature_names), len(importances))
    imp_df = pd.DataFrame({
        'feature'   : all_feature_names[:n],
        'importance': importances[:n]
    }).sort_values('importance', ascending=False)

    top15 = imp_df.head(15)
    plt.figure(figsize=(10, 6))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top15)))
    plt.barh(top15['feature'][::-1], top15['importance'][::-1],
             color=colors, edgecolor='black', linewidth=0.5)
    plt.xlabel('Feature Importance')
    plt.title(f'Top 15 Features — {best_model_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Top 10 features:')
    print(imp_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# CELL 25 — FAIRNESS CHECK BY AGE GROUP
# BUG #14 FIXED: Original code referenced tp_y and fn_y
#                which were never defined anywhere — NameError!
# FIX: Use recall_score() directly on the filtered subset
# ============================================================

print('='*50)
print('FAIRNESS ANALYSIS BY AGE GROUP')
print('='*50)

y_pred_arr  = np.array(y_pred)
y_test_arr  = np.array(y_test)
age_test    = X_test['person_age'].values

young_mask = age_test < 30
old_mask   = age_test >= 50

if young_mask.sum() > 10:
    young_recall = recall_score(           # FIX #14: was tp_y/(tp_y+fn_y)
        y_test_arr[young_mask],
        y_pred_arr[young_mask],
        zero_division=0
    )
    print(f'Recall for age < 30  : {young_recall:.4f}')
else:
    young_recall = None
    print('Age < 30 group too small')

if old_mask.sum() > 10:
    old_recall = recall_score(             # FIX #14
        y_test_arr[old_mask],
        y_pred_arr[old_mask],
        zero_division=0
    )
    print(f'Recall for age >= 50 : {old_recall:.4f}')
else:
    old_recall = None
    print('Age >= 50 group too small')

if young_recall is not None and old_recall is not None:
    diff = abs(young_recall - old_recall)
    print(f'\nRecall difference between groups: {diff:.4f}')
    if diff > 0.1:
        print('WARNING: Potential age-based bias in recall')
    else:
        print('OK: Recall is fairly consistent across age groups')

---
## STEP 11 — Final Summary

In [ ]:
# ============================================================
# CELL 26 — COMPLETE SUMMARY
# ============================================================

print('=' * 70)
print('FINAL SUBMISSION SUMMARY')
print('=' * 70)
print(f'''
BEST MODEL   : {best_model_name}
THRESHOLD    : {best_threshold}

PERFORMANCE METRICS (test set — evaluated ONCE at the end)
------------------------------------------------------------
  ROC-AUC    : {auc_score:.4f}   PRIMARY METRIC
  Recall     : {rec:.4f}   {rec*100:.1f}% of actual defaults caught
  Precision  : {prec:.4f}
  F1-Score   : {f1:.4f}
  Accuracy   : {accuracy:.4f}   (misleading for 96/4 split)
  Brier Score: {brier:.4f}

CONFUSION MATRIX
                  Predicted 0    Predicted 1
  Actual 0   :   {tn:10d}     {fp:10d}   (TN / FP)
  Actual 1   :   {fn:10d}     {tp:10d}   (FN / TP)
''')

print('ALL 14 BUGS FIXED')
print('-'*70)
bugs_fixed = [
  ('#1 ',  'SimpleImputer wrong import         → sklearn.impute'),
  ('#2 ',  'Wrong file path                    → credit_risk_dataset.csv'),
  ('#3 ',  'Post-outcome leakage cols          → dropped before processing'),
  ('#4 ',  'Feature selection before split     → moved after split'),
  ('#5 ',  'Preprocessor fit on train+test     → fit on train only'),
  ('#6 ',  'Hyperparameter tuning on test      → GridSearchCV with CV'),
  ('#7 ',  'Threshold tuning on test           → tuned on validation folds'),
  ('#8 ',  'Normalization stats from full data → train-only stats via pipeline'),
  ('#9 ',  'More leakage cols in section 3     → all dropped'),
  ('#10', 'Wrong column name annual_income   → annual_inc'),
  ('#11', 'auc shadows auc() function        → renamed auc_score'),
  ('#12', 'Missing metric imports            → added precision/recall/brier'),
  ('#13', 'Feature importance after OHE      → get_feature_names_out()'),
  ('#14', 'Undefined tp_y/fn_y              → recall_score() on subset'),
]
for num, desc in bugs_fixed:
    print(f'  Bug {num}: {desc}')

print('\n' + '='*70)
print('WHY ROC-AUC IS THE CORRECT METRIC HERE')
print('='*70)
print('''
  A model that predicts zero defaults for everyone gets 96% accuracy.
  Accuracy is completely misleading for a 96/4 imbalanced dataset.

  ROC-AUC measures discrimination across ALL possible thresholds.
  It is robust to class imbalance and threshold choice.
  ROC-AUC = 0.5 means random guessing.
  ROC-AUC = 1.0 means perfect discrimination.

  --> ROC-AUC is the best metric for this credit risk problem.
''')
print('=' * 70)